# Emergency Debug — Why port 5065 never opens

## Cell 1 — Config

In [3]:
RP_IP    = "192.168.0.99"
SSH_USER = "root"
SSH_PASS = "root"
print("OK")

OK


## Cell 2 — Kill board, add logging, restart manually

In [4]:
import paramiko, time

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS)

# Kill old process
ssh.exec_command("pkill -f RunLock.py")
time.sleep(2)

# Start RunLock.py with output redirected to log file
ssh.exec_command("cd /root && python3 RunLock.py > /tmp/runlock.log 2>&1 &")
time.sleep(5)

# Read what the board printed
_, out, _ = ssh.exec_command("cat /tmp/runlock.log")
log = out.read().decode()
print("=== Board startup log ===")
print(log)

_, out, _ = ssh.exec_command("ss -tlnp | grep -E '5000|5065|5066'")
print("\n=== Open ports ===")
print(out.read().decode())
ssh.close()

=== Board startup log ===
Traceback (most recent call last):
  File "/root/RunLock.py", line 3, in <module>
    from RP_Lock import *
  File "/root/RP_Lock.py", line 54, in <module>
    import rp
ModuleNotFoundError: No module named 'rp'


=== Open ports ===



## Cell 3 — Send start_scan manually via SSH python and watch log

In [5]:
import paramiko, time, socket, struct, json

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS)

print("Sending start_scan command directly to port 5000...")

# Send start_scan via raw socket (bypass all PC code)
content = json.dumps({"action": "start_scan", 
                       "value": {"amplitude": 0.7, "offset": 0.0}},
                     ensure_ascii=False).encode("utf-8")
jh = json.dumps({"byteorder": "little", "content-type": "text/json",
                  "content-encoding": "utf-8", 
                  "content-length": len(content)},
                ensure_ascii=False).encode("utf-8")
raw = struct.pack(">H", len(jh)) + jh + content

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(5)
s.connect((RP_IP, 5000))
s.sendall(raw)
print(f"Sent {len(raw)} bytes to port 5000")
s.setblocking(False)

# Now poll port 5065 every second for 15s
print("Polling port 5065...")
for i in range(15):
    time.sleep(1)
    # Check board log
    _, out, _ = ssh.exec_command("tail -5 /tmp/runlock.log")
    log_tail = out.read().decode().strip()
    # Check port
    _, out2, _ = ssh.exec_command("ss -tlnp | grep -E '5065|5066'")
    ports = out2.read().decode().strip()
    print(f"  t={i+1}s | ports: {ports or 'none'}")
    print(f"         | log: {log_tail.splitlines()[-1] if log_tail else 'empty'}")
    if "5065" in ports:
        print(f"  PORT 5065 OPEN at t={i+1}s!")
        break

_, out, _ = ssh.exec_command("cat /tmp/runlock.log")
print("\n=== Full board log ===")
print(out.read().decode())
s.close()
ssh.close()

Sending start_scan command directly to port 5000...


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

## Cell 4 — Check RP_Lock.py action_start_scan mode guard

In [ ]:
import paramiko

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS)

# Check the mode guard and RP_mode value
_, out, _ = ssh.exec_command(
    "grep -n 'RP_mode\|scan_mon\|only valid\|start_scan\|start_server\|Listening' /root/RP_Lock.py | head -30")
print(out.read().decode())

# Check RunLock.py to see what mode is passed
print("\n=== RunLock.py ===")
_, out, _ = ssh.exec_command("cat /root/RunLock.py")
print(out.read().decode())

ssh.close()